# logX — seq2seq trainer (Colab)

Trains `google/t5-efficient-tiny` on the bundled logX corpus on a GPU.

**Before running:** `Runtime ▸ Change runtime type ▸ T4 GPU` (or any GPU).

1. Run the cells top to bottom.
2. When prompted, upload `logX_colab_bundle.zip` (built by `python -m commons.colab.bundle`).
3. After training it evaluates and lists every failing case (`runs/misses.jsonl`).
4. The last cell downloads `best.zip` (checkpoint + metrics + eval report + misses) — unzip it into your runs dir locally.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU."
!nvidia-smi -L
print(f"torch {torch.__version__}, bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
%pip install -q -U "transformers>=5" accelerate sentencepiece jsonschema

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

if not Path("/content/models-factory/logX/src/train.py").exists():
    print("Upload the colab bundle zip ...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content/models-factory")
    Path(zip_name).unlink()
!find /content/models-factory -maxdepth 3 -type d | sort

In [ ]:
%cd /content/models-factory
!mkdir -p runs
!HF_HUB_OFFLINE=1 python -u -m logX.src.train \
    --model logX/base_model \
    --data-dir logX/clean \
    --batch-size 128 --grad-accum 2 --lr 6e-4 \
    --epochs 15 --patience 3 --early-stop-threshold 0.003 \
    --eval-subset 500 --group-by-length \
    --out-dir runs/colab 2>&1 | tee runs/train_colab.log

## evaluate + list every failing case
Scores test/test_ood (and val), and writes **every** mismatch to `runs/misses.jsonl` (split/input/gold/pred/tags) so you can eyeball or re-run the failures. Per-tag EM in the log shows which categories to fix next.

In [ ]:
%cd /content/models-factory
!HF_HUB_OFFLINE=1 python -u -m logX.src.evaluate \
    --model-dir runs/colab/best \
    --data-dir logX/clean \
    --splits val,test,test_ood \
    --misses-out runs/misses.jsonl 2>&1 | tee runs/eval_colab.log

In [ ]:
import json
misses = [json.loads(l) for l in open("runs/misses.jsonl")]
print(f"{len(misses)} failing cases\n")
for m in misses[:40]:
    print(f"[{m['split']}] tags={m.get('tags')}")
    print(f"  input: {m['input']}")
    print(f"  gold : {m['gold']}")
    print(f"  pred : {m['pred']}\n")
# re-run one by hand:  !python -m logX.src.infer "your text" --model-dir runs/colab/best

In [ ]:
!cd /content/models-factory && zip -qr /content/best.zip \
    runs/colab/best runs/colab/final_metrics.json runs/colab/eval_report.json \
    runs/misses.jsonl runs/train_colab.log runs/eval_colab.log
from google.colab import files
files.download("/content/best.zip")